# Advanced Transformer-Based rPPG Pipeline — v4 (UBFC-rPPG Debug-Fixed Edition)

## What Was Fixed (v4 — Bug Analysis Pass)
| # | Bug | Root Cause | Fix |
|---|-----|-----------|-----|
| A | `negative_pearson` mean biased by padding zeros | `preds * mask` zeros out padding, but `mean(dim=1)` divides by full SEQ_LEN (300) not by valid count → DC-offset in Pearson numerator | Compute mean **only over valid positions** using `sum / num_valid` |
| B | `enhanced_spectral_loss` ignores validity mask | FFT is applied to full (B,T) tensor including zero-padded frames → spectral energy is diluted / shifted for short clips | Per-sample crop to valid length before FFT |
| C | `mean_absolute_bpm_error` ignores mask | `compute_bpm` called on padded (300-sample) array → zero-padding creates spectral artifacts; GT signal is also bandpass-filtered a second time | Accept `masks` param; crop each signal to valid length before FFT |
| D | `BatchNorm1d` in `temporal_conv` | Running stats unstable at val batch_size=1; BN in eval mode uses stale running mean/var | Replace with `GroupNorm(8, D_MODEL)` — instance-independent, works at any batch size |
| E | `validate()` does not collect masks | Masks are never forwarded to `mean_absolute_bpm_error` → BPM computed on zero-padded tensors; no per-subject diagnostic output | Collect all masks; pass to MAE; print per-subject BPM table |
| F | `smooth_l1_loss` loss not normalised by valid count | Denominator is always 300 × B; padding zeros contribute 0 to numerator but inflate the denominator, making the loss scale vary with pad ratio | Divide by `mask.sum()` instead of total elements |
| G | Train/val split not shuffled | `sorted(glob(...))` returns filesystem order; adjacent subjects often share recording conditions → val set may be statistically different from train | Shuffle with a fixed seed before splitting |

## Earlier fixes (v3, v2) inherited unchanged
See original changelog at the top of v3.

In [1]:
# Install dependencies (run once)
# !pip install torch torchvision torchaudio
# !pip install mediapipe opencv-python scipy numpy matplotlib tqdm


## GPU / CPU Detection

This cell **must run first**. It detects whether a CUDA-capable GPU is available and prints full hardware info. If no GPU is found the pipeline automatically falls back to CPU with adjusted settings (smaller batch, fewer default epochs, gradient checkpointing disabled).

In [2]:
import torch
import platform
import multiprocessing

print("=" * 65)
print("   DEVICE CONFIGURATION")
print("=" * 65)

if torch.cuda.is_available():
    _DEVICE = "cuda"
    n_gpus = torch.cuda.device_count()
    print(f"  ✅  GPU Available — {n_gpus} device(s) found")
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        mem_gb = props.total_memory / (1024 ** 3)
        print(f"")
        print(f"  GPU {i}  : {props.name}")
        print(f"           Memory   : {mem_gb:.1f} GB")
        print(f"           CUDA Cap : compute {props.major}.{props.minor}")
        print(f"           SM count : {props.multi_processor_count}")
    print(f"")
    print(f"  PyTorch version : {torch.__version__}")
    print(f"  CUDA version    : {torch.version.cuda}")
    try:
        print(f"  cuDNN version   : {torch.backends.cudnn.version()}")
    except Exception:
        pass
    print(f"")
    print(f"  ► Training will run on  : GPU (cuda)")
else:
    _DEVICE = "cpu"
    cpu_count = multiprocessing.cpu_count()
    print(f"  ⚠️   No CUDA-capable GPU detected — falling back to CPU")
    print(f"")
    print(f"  CPU cores (logical) : {cpu_count}")
    print(f"  Platform            : {platform.processor() or platform.machine()}")
    print(f"  PyTorch version     : {torch.__version__}")
    print(f"")
    print(f"  NOTE: CPU training is significantly slower than GPU.")
    print(f"  • Default epochs are reduced to 10 for CPU (change in CFG).")
    print(f"  • Batch size is set to 1 on CPU to avoid OOM.")
    print(f"  • Gradient checkpointing is disabled on CPU (no benefit).")
    print(f"  • For full training, consider Google Colab (free T4 GPU).")
    print(f"")
    print(f"  ► Training will run on  : CPU")

print("=" * 65)


   DEVICE CONFIGURATION
  ⚠️   No CUDA-capable GPU detected — falling back to CPU

  CPU cores (logical) : 40
  Platform            : Intel64 Family 6 Model 143 Stepping 8, GenuineIntel
  PyTorch version     : 2.3.1+cpu

  NOTE: CPU training is significantly slower than GPU.
  • Default epochs are reduced to 10 for CPU (change in CFG).
  • Batch size is set to 1 on CPU to avoid OOM.
  • Gradient checkpointing is disabled on CPU (no benefit).
  • For full training, consider Google Colab (free T4 GPU).

  ► Training will run on  : CPU


## Imports

In [3]:
import os
import cv2
import glob
import math
import torch
import random
import mediapipe as mp
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from scipy.signal import butter, filtfilt, resample as scipy_resample
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint


## Configuration

Key fields to edit before running:

| Field | Default | Notes |
|-------|---------|-------|
| `UBFC_ROOT` | `./UBFC-rPPG/dataset2` | Path to UBFC-rPPG dataset2 root |
| `GT_FILENAME` | `ground_truth.txt` | Ground-truth filename inside each subject folder |
| `EPOCHS` | 100 (GPU) / 10 (CPU) | Auto-reduced on CPU for quick testing |
| `BATCH_SIZE` | 2 (GPU) / 1 (CPU) | Auto-reduced on CPU to avoid OOM |

UBFC-rPPG dataset2 expected structure:
```
UBFC_ROOT/
  subject1/
    vid.avi
    ground_truth.txt
  subject2/
    vid.avi
    ground_truth.txt
  ...
```

In [4]:
class CFG:
    # ── Device (auto-detected) ────────────────────────────────────────────
    DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

    # ── Model architecture ────────────────────────────────────────────────
    IMG_SIZE     = 128          # spatial size of face-ROI crop (pixels)
    SEQ_LEN      = 300          # fixed temporal window (frames) for padding/cropping
    D_MODEL      = 128          # Transformer embedding dimension
    NHEAD        = 8            # Transformer attention heads
    NUM_LAYERS   = 4            # Transformer encoder layers
    DIM_FF       = 512          # Feed-forward dim (FIX #8: 512 not 2048)
    DROPOUT      = 0.1

    # ── Training ─────────────────────────────────────────────────────────
    BATCH_SIZE   = 2   if torch.cuda.is_available() else 1
    EPOCHS       = 100 if torch.cuda.is_available() else 10
    LR           = 1e-4
    GRAD_CKPT    = torch.cuda.is_available()  # only meaningful on GPU

    # ── Data / video ──────────────────────────────────────────────────────
    FPS          = 30.0         # UBFC-rPPG dataset2 is recorded at 30 fps
    MIN_FRAMES   = 64           # minimum valid (face-detected) frames per clip
    NUM_WORKERS  = 0            # keep 0 for compatibility (esp. Windows/CPU)

    # ── UBFC-rPPG dataset ─────────────────────────────────────────────────
    UBFC_ROOT = r"C:\\Users\\UIET\\Desktop\\physformer\\dataset"
    GT_FILENAME  = "ground_truth.txt"
    TRAIN_RATIO  = 0.8          # 80 % train / 20 % validation (by subject)

    # ── FIX G: reproducible subject-level shuffle before splitting ────────
    # Change this integer to get a different train/val subject assignment.
    # Set to None to disable shuffling (not recommended).
    SPLIT_SEED   = 42

    # ── Checkpoints ───────────────────────────────────────────────────────
    SAVE_BEST    = "best_rppg_ubfc.pth"
    SAVE_FINAL   = "final_rppg_ubfc.pth"

cfg = CFG()
print(f"Configuration loaded.")
print(f"  Device      : {cfg.DEVICE.upper()}")
print(f"  Batch size  : {cfg.BATCH_SIZE}")
print(f"  Epochs      : {cfg.EPOCHS}")
print(f"  Grad ckpt   : {cfg.GRAD_CKPT}")
print(f"  UBFC root   : {cfg.UBFC_ROOT}")
print(f"  Split seed  : {cfg.SPLIT_SEED}")


Configuration loaded.
  Device      : CPU
  Batch size  : 1
  Epochs      : 10
  Grad ckpt   : False
  UBFC root   : C:\\Users\\UIET\\Desktop\\physformer\\dataset
  Split seed  : 42


## FaceMesh — Thread-Safe Initialization

**FIX #3:** A per-process registry avoids pickling the MediaPipe object across DataLoader workers (`num_workers > 0` would otherwise crash).

In [5]:
# Per-process FaceMesh registry  — do NOT pickle this dict across workers
_face_mesh_registry: dict = {}

def get_face_mesh() -> mp.solutions.face_mesh.FaceMesh:
    """Return a FaceMesh instance local to the current OS process."""
    pid = os.getpid()
    if pid not in _face_mesh_registry:
        _face_mesh_registry[pid] = mp.solutions.face_mesh.FaceMesh(
            static_image_mode=False,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.7,
            min_tracking_confidence=0.7,
        )
    return _face_mesh_registry[pid]


def worker_init_fn(worker_id: int) -> None:
    """Called by DataLoader in each worker process to pre-warm FaceMesh."""
    get_face_mesh()


## Landmark Index Groups

Three facial regions are used for ROI extraction: forehead, left cheek, right cheek.

In [6]:
# MediaPipe FaceMesh landmark indices for three ROI regions
FOREHEAD_IDXS = [10, 67, 103, 109, 338, 297]
LEFT_CHEEK    = [50, 187, 205, 207]
RIGHT_CHEEK   = [280, 425, 411, 427]


## Skin Segmentation + ROI Extraction

**IMPROVEMENT — Skin Segmentation:** A YCrCb-based skin mask is applied inside each ROI before computing the mean, suppressing non-skin pixels (hair, teeth, background).

In [7]:
def skin_mask_ycrcb(roi_bgr: np.ndarray) -> np.ndarray:
    """Return a binary mask (0/255) selecting skin pixels in a BGR crop."""
    ycrcb = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2YCrCb)
    mask  = cv2.inRange(ycrcb,
                        np.array([0,   133, 77],  dtype=np.uint8),
                        np.array([255, 173, 127], dtype=np.uint8))
    return mask


def extract_roi(
    frame_rgb: np.ndarray,
    landmarks,
    indices: list,
    apply_skin_mask: bool = True,
) -> tuple:
    """
    Extract a facial ROI given MediaPipe landmark indices.

    Returns
    -------
    roi_resized : np.ndarray (IMG_SIZE, IMG_SIZE, 3) float32 [0,1]
    rgb_mean    : np.ndarray (3,) mean RGB of skin pixels inside ROI
    """
    h, w, _ = frame_rgb.shape
    pts = np.array(
        [(int(landmarks.landmark[i].x * w),
          int(landmarks.landmark[i].y * h)) for i in indices],
        dtype=np.int32,
    )

    geo_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillConvexPoly(geo_mask, pts, 255)

    x, y, bw, bh = cv2.boundingRect(pts)
    if bw == 0 or bh == 0:
        return None, None

    roi_rgb   = frame_rgb[y:y+bh, x:x+bw]
    mask_crop = geo_mask[y:y+bh, x:x+bw]

    if apply_skin_mask:
        roi_bgr  = cv2.cvtColor(roi_rgb, cv2.COLOR_RGB2BGR)
        skin     = skin_mask_ycrcb(roi_bgr)
        combined = cv2.bitwise_and(mask_crop, skin)
    else:
        combined = mask_crop

    skin_pixels = roi_rgb[combined > 0]
    if len(skin_pixels) == 0:
        rgb_mean = roi_rgb.reshape(-1, 3).mean(axis=0).astype(np.float32)
    else:
        rgb_mean = skin_pixels.mean(axis=0).astype(np.float32)

    roi_resized = cv2.resize(roi_rgb, (cfg.IMG_SIZE, cfg.IMG_SIZE))
    roi_resized = roi_resized.astype(np.float32) / 255.0
    return roi_resized, rgb_mean


## Signal Processing

`bandpass_filter` and `compute_bpm` are retained for metric computation. The POS signal estimator is no longer used as a training target (replaced by UBFC ground truth), but is kept for reference.

In [8]:
def POS_WANG(rgb_signals: list) -> np.ndarray:
    """
    Plain Orthogonal Subspace (POS) — Wang et al. 2017.
    Kept for reference / inference-time BPM estimation.
    NOT used as a training target in the UBFC pipeline (GT BVP is used instead).
    """
    rgb  = np.asarray(rgb_signals, dtype=np.float32)   # (N, 3)
    mean = np.mean(rgb, axis=0)
    rgb  = rgb / (mean + 1e-8)
    proj = np.array([[0, 1, -1], [-2, 1, 1]], dtype=np.float32)
    S    = proj @ rgb.T                                 # (2, N)
    alpha = np.std(S[0]) / (np.std(S[1]) + 1e-8)
    return S[0] + alpha * S[1]                          # (N,)


def bandpass_filter(signal: np.ndarray, fs: float = 30.0) -> np.ndarray:
    """
    5th-order Butterworth bandpass 0.75–3.0 Hz (45–180 BPM).
    Returns z-score normalised signal.
    """
    lowcut, highcut = 0.75, 3.0
    nyquist = fs * 0.5
    b, a = butter(5, [lowcut / nyquist, highcut / nyquist], btype='band')
    min_len = 3 * max(len(a), len(b))
    if len(signal) < min_len:
        sig = signal - signal.mean()
        std = signal.std()
        return sig / (std + 1e-8)
    filtered = filtfilt(b, a, signal)
    return ((filtered - filtered.mean()) / (filtered.std() + 1e-8)).astype(np.float32)


def compute_bpm(signal: np.ndarray, fs: float = 30.0) -> float:
    """Dominant frequency (in BPM) within the physiological band."""
    signal  = bandpass_filter(signal, fs)
    freqs   = np.fft.rfftfreq(len(signal), d=1.0 / fs)
    fft_mag = np.abs(np.fft.rfft(signal))
    mask    = (freqs >= 0.75) & (freqs <= 3.0)
    return float(freqs[mask][np.argmax(fft_mag[mask])] * 60.0)


## Motion Augmentation

**IMPROVEMENT:** Applied at the frame-sequence level during training only. Augmentations are applied in sync to both frames and the GT signal so alignment is preserved.

In [9]:
def augment_sequence(
    frames: np.ndarray,
    signal: np.ndarray,
    training: bool = True,
) -> tuple:
    """
    Jointly augment frames (T, H, W, 3) and signal (T,).

    Augmentations (training only):
      • Random horizontal flip            (p = 0.5)
      • Random brightness / contrast jitter
      • Random temporal jitter — drop 0–3 frames from the start (synced)
    """
    if not training:
        return frames, signal

    # Horizontal flip — does not affect rPPG signal
    if random.random() < 0.5:
        frames = frames[:, :, ::-1, :].copy()

    # Brightness / contrast — small perturbation
    alpha = random.uniform(0.8, 1.2)    # contrast scale
    beta  = random.uniform(-0.05, 0.05) # brightness offset
    frames = np.clip(alpha * frames + beta, 0.0, 1.0).astype(np.float32)

    # Temporal jitter — drop a few frames from the start (synced)
    jitter = random.randint(0, 3)
    if jitter > 0 and len(frames) > jitter:
        frames = frames[jitter:]
        signal = signal[jitter:]

    return frames, signal


## Temporal Difference

Computes the frame-to-frame difference for the motion stream. Output length = N − 1.

In [10]:
def temporal_difference(frames: np.ndarray) -> np.ndarray:
    """Frame-to-frame difference.  Output shape: (T-1, H, W, 3)."""
    return (frames[1:].astype(np.float32) - frames[:-1].astype(np.float32))


## UBFC-rPPG Ground Truth Loader *(v3 — New)*

Replaces the POS-estimated signal with the **actual BVP waveform** stored in each subject's `ground_truth.txt`.

### ground_truth.txt format (UBFC-rPPG dataset2)
Each row contains **3 space-separated values**:
```
<BVP_waveform>   <SpO2>   <HR_bpm>
```
- **Column 0** — raw blood-volume-pulse waveform captured by a CMS50E pulse oximeter. This is our training target.
- Columns 1–2 are ignored.

The file is sampled at the same frame rate as the video (~30 fps), so its length closely matches the video frame count. Any minor mismatch is handled by `scipy.signal.resample`.

### Alignment with temporal_difference
`process_video` returns `N−1` frames (after frame-differencing). `load_ubfc_gt` returns a signal of exactly `target_length = N−1` so arrays are always in sync.

In [11]:
def load_ubfc_gt(
    gt_path:       str,
    target_length: int,
    fs:            float = 30.0,
) -> np.ndarray:
    """
    Load the BVP ground-truth signal from a UBFC-rPPG ground_truth.txt file.

    Parameters
    ----------
    gt_path       : absolute path to ground_truth.txt
    target_length : desired output length = n_valid_frames - 1
                    (one less because temporal_difference shrinks by 1)
    fs            : sampling frequency in Hz (default 30)

    Returns
    -------
    signal : np.ndarray, shape (target_length,), float32
             Bandpass-filtered and z-score normalised BVP waveform.

    UBFC-rPPG ground_truth.txt format:
      Rows: <bvp_value>  <spo2>  <hr_bpm>   (space-separated)
      Only column 0 (BVP waveform) is used.
    """
    if not os.path.isfile(gt_path):
        raise FileNotFoundError(
            f"Ground-truth file not found:\n"
            f"  {gt_path}\n"
            f"Check cfg.UBFC_ROOT and cfg.GT_FILENAME."
        )

    raw = np.loadtxt(gt_path, dtype=np.float64)

    # Support both 1-D (single column) and 2-D (multi-column) files
    if raw.ndim == 2:
        bvp = raw[:, 0].astype(np.float32)   # column 0 = BVP waveform
    else:
        bvp = raw.astype(np.float32)          # already 1-D

    # Resample to target_length if lengths differ
    # (handles minor fps-induced mismatches between GT and valid frame count)
    if len(bvp) != target_length:
        bvp = scipy_resample(bvp, target_length).astype(np.float32)

    # Bandpass filter (0.75–3.0 Hz) + z-score normalisation
    signal = bandpass_filter(bvp, fs=fs)
    return signal.astype(np.float32)


## Video Frame Extraction *(v3 — Refactored)*

`process_video` now **returns frames only** — it no longer computes a POS-estimated signal.  
The ground-truth BVP signal is loaded separately by `load_ubfc_gt` and passed in by `RPPGDataset`.

This gives cleaner separation of concerns and enables proper supervised training with real labels.

In [12]:
def process_video(video_path: str) -> tuple:
    """
    Extract dual-stream frame data from a UBFC-rPPG video file.

    Differs from v2: signal generation (POS) is removed.
    The UBFC ground-truth BVP is loaded separately via load_ubfc_gt().

    Parameters
    ----------
    video_path : path to vid.avi (or any supported video file)

    Returns
    -------
    diff_frames    : np.ndarray  (N-1, H, W, 3) float32
                     Normalised frame-to-frame difference (motion stream).
    raw_frames     : np.ndarray  (N-1, H, W, 3) float32
                     Raw face-ROI crops aligned to diff_frames (appearance stream).
    n_valid_frames : int
                     Number of valid (face-detected) frames before differencing.
                     Pass as `n_valid_frames` to load_ubfc_gt().

    Raises
    ------
    FileNotFoundError : if video_path is not a file
    ValueError        : if fewer than cfg.MIN_FRAMES valid frames are found
    """
    if not os.path.isfile(video_path):
        raise FileNotFoundError(
            f"video_path must point to a VIDEO FILE, not a directory.\n"
            f"  Got : {video_path}\n"
            f"  Fix : cfg.UBFC_ROOT/<subject>/vid.avi"
        )

    cap       = cv2.VideoCapture(video_path)
    face_mesh = get_face_mesh()   # FIX #3 — per-process instance

    raw_frames_list: list = []

    while True:
        ret, frame_bgr = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        results   = face_mesh.process(frame_rgb)

        if not results.multi_face_landmarks:     # FIX #5 — skip no-face frames
            continue

        landmarks = results.multi_face_landmarks[0]
        roi_imgs  = []

        for indices in [FOREHEAD_IDXS, LEFT_CHEEK, RIGHT_CHEEK]:
            roi_img, _ = extract_roi(frame_rgb, landmarks, indices)
            if roi_img is not None:
                roi_imgs.append(roi_img)

        if len(roi_imgs) == 0:
            continue

        # FIX #1 — average ROIs → one face image per frame
        face_img = np.mean(np.stack(roi_imgs), axis=0).astype(np.float32)
        raw_frames_list.append(face_img)

    cap.release()

    n_valid_frames = len(raw_frames_list)
    if n_valid_frames < cfg.MIN_FRAMES:
        raise ValueError(
            f"Only {n_valid_frames} valid frames in:\n  {video_path}\n"
            f"Minimum required: {cfg.MIN_FRAMES}.  "
            f"Check the video file and face-detection thresholds."
        )

    raw_arr    = np.array(raw_frames_list)            # (N, H, W, 3)

    # FIX #2 — temporal difference; align raw_frames to same length
    diff_arr   = temporal_difference(raw_arr)          # (N-1, H, W, 3)
    raw_arr    = raw_arr[:-1]                          # (N-1, H, W, 3)

    # Normalise diff frames (zero-mean, unit-variance per video)
    mean = diff_arr.mean(axis=(0, 1, 2), keepdims=True)
    std  = diff_arr.std(axis=(0, 1, 2),  keepdims=True)
    diff_arr = (diff_arr - mean) / (std + 1e-8)

    return diff_arr, raw_arr, n_valid_frames


## UBFC-rPPG Dataset Class *(v3 — Updated)*

`RPPGDataset` now accepts **paired `(video_path, gt_path)` lists**.  
Ground-truth BVP is loaded via `load_ubfc_gt` rather than estimated with POS.

The `__getitem__` pipeline:
1. Extract dual-stream frames from video → `(diff_frames, raw_frames, n_valid)`
2. Load UBFC BVP signal → `signal` of length `n_valid − 1`
3. Jointly augment frames + signal (training mode only)
4. Pad or crop to `cfg.SEQ_LEN`
5. Build a validity mask (1 = real, 0 = padding)

In [13]:
def pad_or_crop(arr: np.ndarray, length: int, axis: int = 0) -> np.ndarray:
    """Trim to `length` along `axis`, or zero-pad if shorter."""
    n = arr.shape[axis]
    if n >= length:
        return np.take(arr, range(length), axis=axis)
    pad_width = [(0, 0)] * arr.ndim
    pad_width[axis] = (0, length - n)
    return np.pad(arr, pad_width, mode='constant', constant_values=0.0)


class RPPGDataset(Dataset):
    """
    UBFC-rPPG supervised dataset.

    Parameters
    ----------
    video_paths : list[str]   — paths to vid.avi files
    gt_paths    : list[str]   — paths to ground_truth.txt (same order)
    training    : bool        — if True, applies motion augmentation
    """

    def __init__(
        self,
        video_paths: list,
        gt_paths:    list,
        training:    bool = True,
    ):
        if len(video_paths) != len(gt_paths):
            raise ValueError(
                f"video_paths ({len(video_paths)}) and gt_paths ({len(gt_paths)}) "
                f"must have the same length."
            )
        self.video_paths = video_paths
        self.gt_paths    = gt_paths
        self.training    = training

    def __len__(self) -> int:
        return len(self.video_paths)

    def __getitem__(self, idx: int) -> tuple:
        video_path = self.video_paths[idx]
        gt_path    = self.gt_paths[idx]

        # ── Step 1: extract frames ─────────────────────────────────────
        diff_frames, raw_frames, n_valid = process_video(video_path)
        # diff_frames, raw_frames : (n_valid-1, H, W, 3)

        # ── Step 2: load UBFC ground-truth BVP signal ──────────────────
        # target_length = n_valid - 1  (aligns with temporal-difference output)
        signal = load_ubfc_gt(gt_path, target_length=n_valid - 1, fs=cfg.FPS)

        # ── Step 3: joint augmentation ─────────────────────────────────
        diff_frames, signal = augment_sequence(diff_frames, signal, training=self.training)
        raw_frames = raw_frames[:len(diff_frames)]   # sync after temporal jitter

        T = len(signal)   # actual (unpadded) sequence length

        # ── Step 4: pad / crop to cfg.SEQ_LEN ─────────────────────────
        # FIX #4 — ensures all samples in a batch have identical shapes
        diff_frames = pad_or_crop(diff_frames, cfg.SEQ_LEN, axis=0)
        raw_frames  = pad_or_crop(raw_frames,  cfg.SEQ_LEN, axis=0)
        signal      = pad_or_crop(signal,      cfg.SEQ_LEN, axis=0)

        # ── Step 5: validity mask ──────────────────────────────────────
        mask = np.zeros(cfg.SEQ_LEN, dtype=np.float32)
        mask[:min(T, cfg.SEQ_LEN)] = 1.0

        return (
            torch.tensor(diff_frames, dtype=torch.float32),   # (SEQ_LEN, H, W, 3)
            torch.tensor(raw_frames,  dtype=torch.float32),   # (SEQ_LEN, H, W, 3)
            torch.tensor(signal,      dtype=torch.float32),   # (SEQ_LEN,)
            torch.tensor(mask,        dtype=torch.float32),   # (SEQ_LEN,)
        )


## Multi-Scale CNN Encoder with Attention Pooling

**IMPROVEMENTS:**
- **Attention Pooling** (`AttentionPool2d`) — learns where to look spatially
- **Multi-Scale** — features at 3 resolutions concatenated → projected to `D_MODEL` (TS-CAN inspired)
- **FIX #7** — `(B*T, C, H, W)` batched CNN call instead of a T-iteration loop

In [14]:
class AttentionPool2d(nn.Module):
    """Spatial soft-attention pooling over (H, W) — learns a saliency map."""

    def __init__(self, in_channels: int):
        super().__init__()
        self.attn = nn.Conv2d(in_channels, 1, kernel_size=1, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x : (B, C, H, W)
        w = torch.softmax(self.attn(x).flatten(2), dim=-1)  # (B, 1, H*W)
        x = x.flatten(2)                                     # (B, C, H*W)
        return (x * w).sum(dim=-1)                           # (B, C)


class MultiScaleSpatialEncoder(nn.Module):
    """
    Dual-stream multi-scale CNN encoder.

    Architecture: shared stem → three parallel branches at different depths.
    Branch feature maps are attention-pooled and concatenated, then projected
    to D_MODEL by a linear layer.

    FIX #7 — called once on the full (B*T, C, H, W) tensor.
    """

    def __init__(self, d_model: int = 128):
        super().__init__()

        def conv_bn_relu(ci, co, k=3, s=1, p=1):
            return nn.Sequential(
                nn.Conv2d(ci, co, k, stride=s, padding=p, bias=False),
                nn.BatchNorm2d(co),
                nn.ReLU(inplace=True),
            )

        # Shared low-level stem
        self.stem = nn.Sequential(
            conv_bn_relu(3, 32),
            conv_bn_relu(32, 32),
            nn.MaxPool2d(2),          # → H/2
        )

        # Scale 1 — shallow
        self.scale1 = nn.Sequential(conv_bn_relu(32, 32))
        self.pool1  = AttentionPool2d(32)

        # Scale 2 — mid
        self.scale2 = nn.Sequential(
            conv_bn_relu(32, 64),
            conv_bn_relu(64, 64),
            nn.MaxPool2d(2),          # → H/4
        )
        self.pool2  = AttentionPool2d(64)

        # Scale 3 — deep
        self.scale3 = nn.Sequential(
            conv_bn_relu(64, 128),
            conv_bn_relu(128, 128),
            nn.MaxPool2d(2),          # → H/8
        )
        self.pool3  = AttentionPool2d(128)

        # Fuse: 32 + 64 + 128 = 224 → d_model
        self.fc = nn.Sequential(
            nn.Linear(32 + 64 + 128, d_model),
            nn.ReLU(inplace=True),
            nn.Dropout(cfg.DROPOUT),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        s  = self.stem(x)                       # (B, 32, H/2, W/2)
        f1 = self.pool1(self.scale1(s))          # (B, 32)
        s2 = self.scale2(s)
        f2 = self.pool2(s2)                      # (B, 64)
        f3 = self.pool3(self.scale3(s2))         # (B, 128)
        return self.fc(torch.cat([f1, f2, f3], dim=-1))   # (B, d_model)


## Positional Encoding

**FIX #6:** Sinusoidal positional encoding — the original transformer was permutation-invariant and could not distinguish temporal order.

In [15]:
class SinusoidalPositionalEncoding(nn.Module):

    def __init__(self, d_model: int, max_len: int = 2000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe  = torch.zeros(max_len, d_model)               # (L, D)
        pos = torch.arange(max_len).unsqueeze(1).float()  # (L, 1)
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))        # (1, L, D)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x : (B, T, D)
        return self.dropout(x + self.pe[:, :x.size(1)])


## Dual-Stream rPPG Transformer

**IMPROVEMENTS:**
- **Dual stream** — separate `MultiScaleSpatialEncoder` for motion (diff) and appearance (raw) branches, fused before the Transformer (TS-CAN inspired)
- **FIX #6** — Sinusoidal positional encoding
- **FIX #7** — Batched CNN via `(B*T, C, H, W)` reshape
- **FIX #8** — `dim_feedforward=512`
- **Gradient Checkpointing** — reduces activation memory on GPU

In [16]:
class RPPGTransformer(nn.Module):

    def __init__(self):
        super().__init__()

        # Dual-stream spatial encoders (one per stream)
        self.motion_enc     = MultiScaleSpatialEncoder(cfg.D_MODEL)
        self.appearance_enc = MultiScaleSpatialEncoder(cfg.D_MODEL)

        # Stream fusion: concat(motion, appearance) → D_MODEL
        self.stream_fuse = nn.Sequential(
            nn.Linear(cfg.D_MODEL * 2, cfg.D_MODEL),
            nn.ReLU(inplace=True),
        )

        # FIX #6 — sinusoidal positional encoding
        self.pos_enc = SinusoidalPositionalEncoding(
            cfg.D_MODEL, max_len=cfg.SEQ_LEN + 10, dropout=cfg.DROPOUT
        )

        # Transformer encoder — FIX #8: dim_feedforward=512, Pre-LN for stability
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.D_MODEL,
            nhead=cfg.NHEAD,
            dim_feedforward=cfg.DIM_FF,
            dropout=cfg.DROPOUT,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=cfg.NUM_LAYERS,
            enable_nested_tensor=False,
        )

        # ── BUG D FIX: GroupNorm instead of BatchNorm1d ──────────────────────
        # ORIGINAL BUG:
        #   BatchNorm1d uses running mean/variance that are updated per batch.
        #   When val batch_size=1 (CPU mode default), BN in eval() uses stale
        #   running stats that may not match the single sample's statistics.
        #   On GPU with batch_size=2, BN is still unstable for small batches.
        #
        # FIX: GroupNorm(num_groups=8) normalises within each (D_MODEL/8)-dim
        #   group independently of batch size — correct and stable at B=1.
        #   8 groups for D_MODEL=128 → 16 channels per group (well-conditioned).
        # ─────────────────────────────────────────────────────────────────────
        self.temporal_conv = nn.Sequential(
            nn.Conv1d(cfg.D_MODEL, cfg.D_MODEL, 3, padding=1, bias=False),
            nn.GroupNorm(8, cfg.D_MODEL),   # ← was BatchNorm1d; GroupNorm is
            nn.ReLU(inplace=True),          #   batch-size-independent
            nn.Conv1d(cfg.D_MODEL, cfg.D_MODEL, 3, padding=1, bias=False),
        )

        # Output regression head
        self.head = nn.Linear(cfg.D_MODEL, 1)

    # ── helper ────────────────────────────────────────────────────────────────
    def _encode_stream(
        self,
        encoder: nn.Module,
        x:       torch.Tensor,
    ) -> torch.Tensor:
        """
        FIX #7 — batched CNN call.
        x   : (B, T, H, W, C)
        out : (B, T, D_MODEL)
        """
        B, T, H, W, C = x.shape
        x = x.permute(0, 1, 4, 2, 3).reshape(B * T, C, H, W)  # (B*T, C, H, W)
        if cfg.GRAD_CKPT and self.training:
            feats = checkpoint(encoder, x, use_reentrant=False)
        else:
            feats = encoder(x)
        return feats.view(B, T, -1)                             # (B, T, D_MODEL)

    # ── forward ───────────────────────────────────────────────────────────────
    def forward(
        self,
        diff_frames:          torch.Tensor,
        raw_frames:           torch.Tensor,
        src_key_padding_mask: torch.Tensor = None,
    ) -> torch.Tensor:
        """
        Parameters
        ----------
        diff_frames          : (B, T, H, W, 3) — motion stream
        raw_frames           : (B, T, H, W, 3) — appearance stream
        src_key_padding_mask : (B, T) bool — True where padding (ignored)

        Returns
        -------
        pred : (B, T) — predicted rPPG signal
        """
        m = self._encode_stream(self.motion_enc,     diff_frames)  # (B, T, D)
        a = self._encode_stream(self.appearance_enc, raw_frames)   # (B, T, D)

        x = self.stream_fuse(torch.cat([m, a], dim=-1))            # (B, T, D)
        x = self.pos_enc(x)                                        # (B, T, D)
        x = self.transformer(x, src_key_padding_mask=src_key_padding_mask)

        # Temporal refinement (Conv1D operates on the time dimension)
        x = self.temporal_conv(x.permute(0, 2, 1)).permute(0, 2, 1)  # (B, T, D)

        return self.head(x).squeeze(-1)                            # (B, T)


## Loss Functions

**IMPROVEMENT — Enhanced Spectral Loss:** Restricted to the physiological BPM band (0.75–3.0 Hz) with an additional KL-divergence term that sharpens the spectral peak.

In [17]:
# ── BUG A FIX: negative_pearson ───────────────────────────────────────────────
# ORIGINAL BUG:
#   preds  = preds  * mask          # zeros out padding
#   preds_c = preds - preds.mean()  # mean() divides by full SEQ_LEN (300)!
#   → When padding is large, the mean is pulled toward zero, introducing a
#     systematic DC-offset error in the Pearson numerator and denominator.
#
# FIX: compute mean ONLY over valid (non-padded) positions using
#   sum(valid values) / num_valid_positions
# ─────────────────────────────────────────────────────────────────────────────
def negative_pearson(
    preds:  torch.Tensor,
    labels: torch.Tensor,
    mask:   torch.Tensor = None,
) -> torch.Tensor:
    """
    1 − Pearson correlation coefficient (lower = better aligned waveforms).

    preds, labels : (B, T)
    mask          : (B, T)  1 = valid frame, 0 = padding

    FIX A: mean is computed only over valid frames (mask-weighted sum / count).
    Previously: preds * mask → mean(dim=1) divided by SEQ_LEN (300), not by
    the true valid count — introducing a DC bias proportional to padding ratio.
    """
    if mask is not None:
        # Valid frame count per sample, at least 1 to avoid division by zero
        num_valid = mask.sum(dim=1, keepdim=True).clamp(min=1.0)

        # Masked mean over VALID positions only  ← THE FIX (was .mean() before)
        preds_mean  = (preds  * mask).sum(dim=1, keepdim=True) / num_valid
        labels_mean = (labels * mask).sum(dim=1, keepdim=True) / num_valid

        # Zero-out padding from centred signals
        preds_c  = (preds  - preds_mean)  * mask
        labels_c = (labels - labels_mean) * mask
    else:
        preds_c  = preds  - preds.mean(dim=1,  keepdim=True)
        labels_c = labels - labels.mean(dim=1, keepdim=True)

    num   = torch.sum(preds_c * labels_c, dim=1)
    denom = torch.sqrt(
        torch.sum(preds_c  ** 2, dim=1) *
        torch.sum(labels_c ** 2, dim=1)
    )
    return (1.0 - num / (denom + 1e-8)).mean()


# ── BUG B FIX: enhanced_spectral_loss ─────────────────────────────────────────
# ORIGINAL BUG:
#   FFT was applied to the full (B, T) tensor including zero-padded frames.
#   For clips shorter than SEQ_LEN the padded zeros:
#     • Dilute spectral energy (peak magnitude drops)
#     • Create a fake low-frequency component from the signal→zero discontinuity
#     • Make the KL term push the model toward the wrong frequency
#
# FIX: per-sample crop to the true valid length before computing FFT.
#   We loop over the batch (small B in rPPG training) and process each sample
#   at its own valid length. This preserves exact frequency resolution.
# ─────────────────────────────────────────────────────────────────────────────
def _spectral_loss_single(
    pred_1d:   torch.Tensor,   # (T_valid,)
    target_1d: torch.Tensor,   # (T_valid,)
    fs:        float = 30.0,
) -> torch.Tensor:
    """Spectral L1 + KL for a single (unpadded) signal pair."""
    T     = pred_1d.shape[0]
    freqs = torch.fft.rfftfreq(T, d=1.0 / fs).to(pred_1d.device)
    band  = (freqs >= 0.75) & (freqs <= 3.0)

    pred_mag = torch.abs(torch.fft.rfft(pred_1d  ))[band]   # (F_band,)
    gt_mag   = torch.abs(torch.fft.rfft(target_1d))[band]

    l1 = torch.mean(torch.abs(pred_mag - gt_mag))

    pred_norm = pred_mag / (pred_mag.sum() + 1e-8)
    gt_norm   = gt_mag   / (gt_mag.sum()   + 1e-8)
    kl = F.kl_div(torch.log(pred_norm + 1e-8), gt_norm, reduction='sum')

    return l1 + 0.1 * kl


def enhanced_spectral_loss(
    pred:   torch.Tensor,
    target: torch.Tensor,
    mask:   torch.Tensor = None,   # ← NEW: (B, T) validity mask
    fs:     float = 30.0,
) -> torch.Tensor:
    """
    Frequency-domain loss restricted to physiological BPM band (0.75–3.0 Hz).
    Combines L1 magnitude difference + KL-divergence on normalised spectrum.

    pred, target : (B, T)
    mask         : (B, T) — 1 = valid, 0 = padding

    FIX B: per-sample crop to valid length before FFT.
    Previously: FFT was run on the full padded (B,T) tensor — zero-padding
    distorts the spectrum and corrupts the KL term.
    """
    if mask is None:
        # No mask → treat all frames as valid (backward-compatible)
        T     = pred.shape[1]
        freqs = torch.fft.rfftfreq(T, d=1.0 / fs).to(pred.device)
        band  = (freqs >= 0.75) & (freqs <= 3.0)
        pred_mag = torch.abs(torch.fft.rfft(pred,   dim=1))[:, band]
        gt_mag   = torch.abs(torch.fft.rfft(target, dim=1))[:, band]
        l1 = torch.mean(torch.abs(pred_mag - gt_mag))
        pred_norm = pred_mag / (pred_mag.sum(dim=1, keepdim=True) + 1e-8)
        gt_norm   = gt_mag   / (gt_mag.sum(  dim=1, keepdim=True) + 1e-8)
        kl = F.kl_div(torch.log(pred_norm + 1e-8), gt_norm, reduction='batchmean')
        return l1 + 0.1 * kl

    # ── With mask: process each sample at its own valid length ── (THE FIX)
    losses = []
    for i in range(pred.shape[0]):
        valid_len = int(mask[i].sum().item())
        if valid_len < 16:          # need at least 16 frames for a stable FFT
            continue
        losses.append(
            _spectral_loss_single(pred[i, :valid_len], target[i, :valid_len], fs)
        )

    if not losses:
        return pred.new_tensor(0.0)   # degenerate batch — return 0 gracefully
    return torch.stack(losses).mean()


## Metrics

In [18]:
# ── BUG C FIX: mean_absolute_bpm_error ───────────────────────────────────────
# ORIGINAL BUG:
#   compute_bpm() was called on the full (SEQ_LEN=300) signal including zero-
#   padded frames. Zeros at the end create a low-frequency artifact after the
#   bandpass filter, shifting the argmax(FFT) toward the wrong bin.
#   Also: the GT signal from DataLoader has already been bandpass-filtered once
#   inside load_ubfc_gt() — calling bandpass_filter() again inside compute_bpm()
#   is redundant (harmless but wastes time).
#
# FIX: accept an optional `masks` parameter; crop each signal to its valid
#   length before calling compute_bpm().  When no mask is provided (backward
#   compatibility) the function behaves exactly as before.
# ─────────────────────────────────────────────────────────────────────────────
def mean_absolute_bpm_error(
    pred_signals: np.ndarray,          # (B, T)
    gt_signals:   np.ndarray,          # (B, T)
    masks:        np.ndarray = None,   # (B, T) — NEW: 1=valid, 0=padding
    fs:           float      = cfg.FPS,
) -> float:
    """
    Mean |predicted BPM − ground-truth BPM| over a batch.

    FIX C: signals are cropped to their valid (non-padded) length before BPM
    extraction. Without cropping, zero-padded tails corrupt the FFT spectrum.
    """
    errors = []
    for i, (p, g) in enumerate(zip(pred_signals, gt_signals)):
        if masks is not None:
            valid_len = int(masks[i].sum())
            if valid_len < 16:      # too short for reliable FFT — skip
                continue
            p = p[:valid_len]
            g = g[:valid_len]

        bpm_p = compute_bpm(p.astype(np.float64), fs)
        bpm_g = compute_bpm(g.astype(np.float64), fs)
        errors.append(abs(bpm_p - bpm_g))

    if not errors:
        return float('nan')
    return float(np.mean(errors))


## Training Setup

Model, optimiser, and scheduler are constructed here. A rough CPU time estimate is printed when no GPU is detected.

In [19]:
model = RPPGTransformer().to(cfg.DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.LR,
    weight_decay=1e-2,
    betas=(0.9, 0.95),
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=cfg.EPOCHS,
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model            : RPPGTransformer (Dual-Stream + Multi-Scale CNN + Transformer)")
print(f"Trainable params : {total_params:,}")
print(f"Training device  : {cfg.DEVICE.upper()}")
print(f"Epochs           : {cfg.EPOCHS}")
print(f"Batch size       : {cfg.BATCH_SIZE}")
print(f"Grad checkpointing: {cfg.GRAD_CKPT}")

if cfg.DEVICE == "cpu":
    print()
    print("⚠️  CPU mode active — training will be slow.")
    print("   Tips to speed up on CPU:")
    print("   • Set cfg.EPOCHS = 3–5 for a quick smoke-test.")
    print("   • Set cfg.SEQ_LEN = 150 to halve sequence processing time.")
    print("   • Use torch.set_num_threads(N) to exploit all CPU cores.")
    import multiprocessing
    torch.set_num_threads(multiprocessing.cpu_count())
    print(f"   • PyTorch threads set to {torch.get_num_threads()}.")


Model            : RPPGTransformer (Dual-Stream + Multi-Scale CNN + Transformer)
Trainable params : 1,576,199
Training device  : CPU
Epochs           : 10
Batch size       : 1
Grad checkpointing: False

⚠️  CPU mode active — training will be slow.
   Tips to speed up on CPU:
   • Set cfg.EPOCHS = 3–5 for a quick smoke-test.
   • Set cfg.SEQ_LEN = 150 to halve sequence processing time.
   • Use torch.set_num_threads(N) to exploit all CPU cores.
   • PyTorch threads set to 40.


## Training & Validation Loops

**FIX #10:** Validation loop with loss + MAE BPM metric and best-model checkpoint. Padding mask is passed to both the model and the Pearson loss.

In [20]:
def train_one_epoch(loader: DataLoader) -> float:
    model.train()
    total_loss = 0.0

    for diff_frames, raw_frames, signal, mask in tqdm(loader, desc="Train", leave=False):
        diff_frames = diff_frames.to(cfg.DEVICE)
        raw_frames  = raw_frames.to(cfg.DEVICE)
        signal      = signal.to(cfg.DEVICE)
        mask        = mask.to(cfg.DEVICE)

        pad_mask = (mask == 0)   # True where padding — passed to Transformer

        optimizer.zero_grad()
        pred = model(diff_frames, raw_frames, src_key_padding_mask=pad_mask)

        # ── NaN guard (catches exploding gradients early) ──────────────────
        if torch.isnan(pred).any() or torch.isinf(pred).any():
            print("⚠️  NaN/Inf in predictions — skipping batch")
            optimizer.zero_grad()
            continue

        pearson = negative_pearson(pred, signal, mask=mask)

        # ── BUG F FIX: normalise smooth_l1 by valid frame count ───────────
        # ORIGINAL BUG:
        #   F.smooth_l1_loss(pred*mask, signal*mask) uses 'mean' reduction,
        #   dividing by B*T=B*300. Padded positions contribute 0 to the
        #   numerator but inflate the denominator → loss underestimated for
        #   short clips by factor (valid_count / 300).
        #
        # FIX: multiply by B*T then divide by actual valid count so the loss
        #   magnitude is independent of the padding ratio.
        # ──────────────────────────────────────────────────────────────────
        valid_count = mask.sum().clamp(min=1.0)
        smooth = (
            F.smooth_l1_loss(pred * mask, signal * mask, reduction='sum')
            / valid_count
        )

        # Pass mask to spectral loss (FIX B)
        spec = enhanced_spectral_loss(pred, signal, mask=mask)

        loss = 0.5 * pearson + 0.3 * smooth + 0.2 * spec

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    return total_loss / max(len(loader), 1)


@torch.no_grad()
def validate(loader: DataLoader) -> tuple:
    """Returns (val_loss, mean_bpm_mae)."""
    model.eval()
    total_loss = 0.0
    all_preds, all_gts, all_masks = [], [], []   # ← collect masks (FIX E)

    for diff_frames, raw_frames, signal, mask in tqdm(loader, desc="Val  ", leave=False):
        diff_frames = diff_frames.to(cfg.DEVICE)
        raw_frames  = raw_frames.to(cfg.DEVICE)
        signal      = signal.to(cfg.DEVICE)
        mask        = mask.to(cfg.DEVICE)
        pad_mask    = (mask == 0)

        pred = model(diff_frames, raw_frames, src_key_padding_mask=pad_mask)

        pearson = negative_pearson(pred, signal, mask=mask)

        # FIX F: normalise by valid count (same as train)
        valid_count = mask.sum().clamp(min=1.0)
        smooth = (
            F.smooth_l1_loss(pred * mask, signal * mask, reduction='sum')
            / valid_count
        )

        # FIX B: pass mask to spectral loss
        spec = enhanced_spectral_loss(pred, signal, mask=mask)

        loss = 0.5 * pearson + 0.3 * smooth + 0.2 * spec
        total_loss += loss.item()

        all_preds.append(pred.cpu().numpy())
        all_gts.append(signal.cpu().numpy())
        all_masks.append(mask.cpu().numpy())    # ← collect mask (FIX E)

    all_preds = np.concatenate(all_preds, axis=0)   # (N_val, T)
    all_gts   = np.concatenate(all_gts,   axis=0)
    all_masks = np.concatenate(all_masks, axis=0)   # ← (N_val, T)

    # ── BUG E FIX: pass masks so BPM is computed only on valid frames ────────
    # ORIGINAL BUG:
    #   mean_absolute_bpm_error(all_preds, all_gts) — masks were never collected
    #   → compute_bpm() ran on the full 300-sample array including padding zeros.
    #   The zero-appended tail distorts the FFT and produces wrong dominant freq.
    # ─────────────────────────────────────────────────────────────────────────
    mae_bpm = mean_absolute_bpm_error(all_preds, all_gts, masks=all_masks, fs=cfg.FPS)

    # ── Per-subject BPM diagnostic (prints first 5 subjects) ─────────────────
    print()
    print(f"  {'Subj':>5} | {'Valid':>5} | {'PredBPM':>7} | {'GT BPM':>7} | {'|Error|':>7}")
    print(f"  {'-----':>5}-+-{'-----':>5}-+-{'-------':>7}-+-{'-------':>7}-+-{'-------':>7}")
    for i in range(min(5, len(all_preds))):
        vlen = int(all_masks[i].sum())
        if vlen >= 16:
            bpm_p = compute_bpm(all_preds[i][:vlen].astype(np.float64), cfg.FPS)
            bpm_g = compute_bpm(all_gts[i][:vlen].astype(np.float64),   cfg.FPS)
            print(f"  {i+1:>5} | {vlen:>5} | {bpm_p:>7.1f} | {bpm_g:>7.1f} | {abs(bpm_p-bpm_g):>7.1f}")
        else:
            print(f"  {i+1:>5} | {vlen:>5} | {'skip':>7} | {'skip':>7} | {'skip':>7}")
    print()

    return total_loss / max(len(loader), 1), mae_bpm


## UBFC-rPPG Dataset Path Discovery *(v3 — New)*

This cell auto-discovers all `vid.avi` + `ground_truth.txt` pairs under `cfg.UBFC_ROOT`.  
Update `cfg.UBFC_ROOT` in the **Configuration** cell before running.

**Expected folder structure:**
```
<UBFC_ROOT>/
  subject1/
    vid.avi
    ground_truth.txt
  subject2/
    vid.avi
    ground_truth.txt
  ...
```

The subjects are split 80 % / 20 % by index (train / validation).  
Shuffle `all_video_paths` before splitting if you want a randomised split.

In [21]:
# ── Auto-discover vid.avi files ───────────────────────────────────────────────
vid_pattern = os.path.join(cfg.UBFC_ROOT, "*", "vid.avi")
all_video_paths = sorted(glob.glob(vid_pattern))

# Fallback: search for .mp4 if no .avi found
if len(all_video_paths) == 0:
    mp4_pattern = os.path.join(cfg.UBFC_ROOT, "*", "*.mp4")
    all_video_paths = sorted(glob.glob(mp4_pattern))

if len(all_video_paths) == 0:
    raise FileNotFoundError(
        f"\n\u274c No video files found under: {cfg.UBFC_ROOT}\n"
        f"   Searched patterns:\n"
        f"     {vid_pattern}\n"
        f"   Please update cfg.UBFC_ROOT in the Configuration cell.\n"
        f"   Expected structure: <UBFC_ROOT>/subject1/vid.avi"
    )

# ── Build paired ground-truth path list ───────────────────────────────────────
all_gt_paths = []
missing_gt   = []

for vp in all_video_paths:
    gt = os.path.join(os.path.dirname(vp), cfg.GT_FILENAME)
    if not os.path.isfile(gt):
        missing_gt.append(gt)
    else:
        all_gt_paths.append(gt)

if missing_gt:
    raise FileNotFoundError(
        f"\n\u274c Ground-truth file(s) missing for {len(missing_gt)} subject(s):\n"
        + "\n".join(f"   {p}" for p in missing_gt[:5])
        + (f"\n   ... and {len(missing_gt)-5} more" if len(missing_gt) > 5 else "")
        + f"\n\nCheck cfg.GT_FILENAME (currently: '{cfg.GT_FILENAME}').")

# Re-filter video paths to only those with valid GT
all_video_paths = [
    vp for vp in all_video_paths
    if os.path.isfile(os.path.join(os.path.dirname(vp), cfg.GT_FILENAME))
]

# ── BUG G FIX: shuffle before train/val split ────────────────────────────────
# ORIGINAL BUG:
#   sorted(glob(...)) returns filesystem order (subject1, subject2, ...).
#   Consecutive subjects often share recording session, lighting, or operator.
#   An unshuffled 80/20 split puts the last ~20% of subjects into val —
#   they may differ systematically from the first 80% (training) subjects.
#
# FIX: shuffle with a fixed random seed for reproducibility.
#   Change SPLIT_SEED in cfg to get a different split.
# ─────────────────────────────────────────────────────────────────────────────
SPLIT_SEED = cfg.SPLIT_SEED  # defined in CFG; default = 42
rng = random.Random(SPLIT_SEED)
combined = list(zip(all_video_paths, all_gt_paths))
rng.shuffle(combined)
all_video_paths, all_gt_paths = zip(*combined)
all_video_paths = list(all_video_paths)
all_gt_paths    = list(all_gt_paths)

# ── Summary ────────────────────────────────────────────────────────────────────
n_total = len(all_video_paths)
print(f"\u2705  Found {n_total} subjects in UBFC-rPPG dataset.")
print()
print("Sample paths (first 5 after shuffle):")
for vp, gp in zip(all_video_paths[:5], all_gt_paths[:5]):
    print(f"  Video : {vp}")
    print(f"  GT    : {gp}")
    print()

# ── Train / validation split (80 % / 20 % by subject, SHUFFLED) ───────────────
n_train = int(cfg.TRAIN_RATIO * n_total)
n_val   = n_total - n_train

train_video_paths = all_video_paths[:n_train]
train_gt_paths    = all_gt_paths[:n_train]
val_video_paths   = all_video_paths[n_train:]
val_gt_paths      = all_gt_paths[n_train:]

print(f"Split  \u2192  Train: {n_train} subjects  |  Val: {n_val} subjects")
print(f"Split seed: {SPLIT_SEED} (change cfg.SPLIT_SEED for a different split)")


✅  Found 42 subjects in UBFC-rPPG dataset.

Sample paths (first 5 after shuffle):
  Video : C:\\Users\\UIET\\Desktop\\physformer\\dataset\subject1\vid.avi
  GT    : C:\\Users\\UIET\\Desktop\\physformer\\dataset\subject1\ground_truth.txt

  Video : C:\\Users\\UIET\\Desktop\\physformer\\dataset\subject13\vid.avi
  GT    : C:\\Users\\UIET\\Desktop\\physformer\\dataset\subject13\ground_truth.txt

  Video : C:\\Users\\UIET\\Desktop\\physformer\\dataset\subject20\vid.avi
  GT    : C:\\Users\\UIET\\Desktop\\physformer\\dataset\subject20\ground_truth.txt

  Video : C:\\Users\\UIET\\Desktop\\physformer\\dataset\subject39\vid.avi
  GT    : C:\\Users\\UIET\\Desktop\\physformer\\dataset\subject39\ground_truth.txt

  Video : C:\\Users\\UIET\\Desktop\\physformer\\dataset\subject49\vid.avi
  GT    : C:\\Users\\UIET\\Desktop\\physformer\\dataset\subject49\ground_truth.txt

Split  →  Train: 33 subjects  |  Val: 9 subjects
Split seed: 42 (change cfg.SPLIT_SEED for a different split)


## DataLoaders

`pin_memory` is enabled only when training on GPU (it has no benefit on CPU and wastes memory). The `worker_init_fn` pre-warms a per-process FaceMesh instance in each worker.

In [22]:
train_dataset = RPPGDataset(train_video_paths, train_gt_paths, training=True)
val_dataset   = RPPGDataset(val_video_paths,   val_gt_paths,   training=False)

_pin = (cfg.DEVICE == "cuda")   # pin_memory only useful with GPU

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=cfg.NUM_WORKERS,
    worker_init_fn=worker_init_fn,
    pin_memory=_pin,
    drop_last=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    worker_init_fn=worker_init_fn,
    pin_memory=_pin,
    drop_last=False,
)

print(f"Train loader : {len(train_loader):3d} batches  "
      f"(batch_size={cfg.BATCH_SIZE}, device={cfg.DEVICE.upper()})")
print(f"Val   loader : {len(val_loader):3d} batches")
print()
if cfg.DEVICE == "cpu":
    est_secs_per_epoch = len(train_loader) * cfg.BATCH_SIZE * 15   # rough estimate
    print(f"⚠️  Rough CPU time estimate: ~{est_secs_per_epoch:.0f}s per epoch  "
          f"({est_secs_per_epoch * cfg.EPOCHS / 60:.0f} min total for {cfg.EPOCHS} epochs).")
    print("   This varies widely with video length and CPU speed.")


Train loader :  33 batches  (batch_size=1, device=CPU)
Val   loader :   9 batches

⚠️  Rough CPU time estimate: ~495s per epoch  (82 min total for 10 epochs).
   This varies widely with video length and CPU speed.


## Start Training

Saves `best_rppg_ubfc.pth` whenever validation MAE BPM improves. Final model saved as `final_rppg_ubfc.pth` regardless.

In [ ]:
best_mae   = float("inf")
best_epoch = 0

for epoch in range(cfg.EPOCHS):
    train_loss = train_one_epoch(train_loader)

    if len(val_video_paths) > 0:
        val_loss, mae_bpm = validate(val_loader)
        print(
            f"Epoch {epoch+1:3d}/{cfg.EPOCHS}"
            f" | Train Loss: {train_loss:.4f}"
            f" | Val Loss: {val_loss:.4f}"
            f" | MAE BPM: {mae_bpm:.2f}"
        )

        if mae_bpm < best_mae:
            best_mae   = mae_bpm
            best_epoch = epoch + 1
            torch.save(model.state_dict(), cfg.SAVE_BEST)
            print(f"           ✅  New best MAE {best_mae:.2f} BPM — model saved → {cfg.SAVE_BEST}")
    else:
        print(f"Epoch {epoch+1:3d}/{cfg.EPOCHS} | Train Loss: {train_loss:.4f}")

torch.save(model.state_dict(), cfg.SAVE_FINAL)
print()
print(f"Training complete.")
print(f"  Best MAE : {best_mae:.2f} BPM  (epoch {best_epoch})")
print(f"  Saved    : {cfg.SAVE_FINAL}  (final)")
print(f"           : {cfg.SAVE_BEST}   (best val MAE)")


Train:   0%|                                                                                    | 0/33 [00:00<?, ?it/s]C:\Users\UIET\anaconda3\envs\nrppg\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '



   Subj | Valid | PredBPM |  GT BPM | |Error|
  ------+-------+---------+---------+--------
      1 |   300 |    60.0 |    48.0 |    12.0
      2 |   300 |    60.0 |    48.0 |    12.0
      3 |   300 |    60.0 |    48.0 |    12.0
      4 |   300 |    60.0 |    48.0 |    12.0
      5 |   300 |    60.0 |    48.0 |    12.0

Epoch   1/10 | Train Loss: 8.6980 | Val Loss: 4.9789 | MAE BPM: 12.00
           ✅  New best MAE 12.00 BPM — model saved → best_rppg_ubfc.pth


Train:  27%|████████████████████▋                                                       | 9/33 [03:58<11:31, 28.80s/it]

## Self-Supervised Pre-training Hook *(Future Improvement)*

Stub for contrastive / masked-autoencoder pre-training on unlabelled face videos. Attach `ContrastiveRPPGHead` to a frozen `RPPGTransformer` backbone and train with a contrastive loss (e.g. SimCLR / BYOL) before fine-tuning.

In [ ]:
class ContrastiveRPPGHead(nn.Module):
    """
    MLP projection head for self-supervised pre-training.

    Usage (pseudo-code)
    -------------------
    backbone = RPPGTransformer()         # remove .head for feature extraction
    proj     = ContrastiveRPPGHead(cfg.D_MODEL)
    z        = proj(backbone_features)  # (B, proj_dim)
    loss     = nt_xent_loss(z_a, z_b)  # SimCLR / BYOL objective
    """

    def __init__(self, d_model: int = 128, proj_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(inplace=True),
            nn.Linear(d_model, proj_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x : (B, T, D)  → mean-pool over time → (B, D) → project
        return self.net(x.mean(dim=1))

print("ContrastiveRPPGHead stub ready for self-supervised pre-training.")


# Summary of All Changes

## v4 Bugs Fixed (this version)
| # | Bug | Root Cause | Fix |
|---|-----|-----------|-----|
| A | `negative_pearson` mean biased by padding | `.mean(dim=1)` divided by SEQ_LEN (300), not valid count | Compute mean via `(x*mask).sum() / num_valid` |
| B | `enhanced_spectral_loss` ignores mask | FFT over zero-padded tail shifts spectral peak | Per-sample crop to valid length before FFT |
| C | `mean_absolute_bpm_error` ignores mask | `compute_bpm` on padded signal → wrong dominant frequency | Accept `masks` param; crop before calling `compute_bpm` |
| D | `BatchNorm1d` in `temporal_conv` | Running stats unreliable at val batch_size=1 | Replaced with `GroupNorm(8, D_MODEL)` |
| E | `validate()` never collected masks | MAE always computed on padded tensors; no BPM diagnostics | Collect masks; pass to `mean_absolute_bpm_error`; print table |
| F | `smooth_l1_loss` denominator includes padding | `reduction='mean'` divides by B×SEQ_LEN, not valid count | Switch to `reduction='sum'` then divide by `mask.sum()` |
| G | Train/val split not shuffled | `sorted(glob)` → consecutive subjects in val; possible bias | `random.shuffle` with `cfg.SPLIT_SEED=42` before splitting |

## v2 Bugs Fixed (inherited)
| # | Bug | Fix |
|---|-----|-----|
| 1 | `rgb_signals` had 3× too many entries | Average 3 ROI means → 1 per frame |
| 2 | Signal–frame length mismatch after `temporal_difference` | Trim signal by 1 after differencing |
| 3 | Global `face_mesh` not picklable → DataLoader crash | Per-process registry + `worker_init_fn` |
| 4 | Variable sequence lengths crash `collate_fn` | `pad_or_crop` to `SEQ_LEN` + validity mask |
| 5 | No guard for empty face detection | `ValueError` if < `MIN_FRAMES` valid frames |
| 6 | No positional encoding in Transformer | Sinusoidal PE added |
| 7 | Frame-by-frame CNN loop → slow / OOM | Reshape to `(B*T, C, H, W)`, single batched call |
| 8 | `dim_feedforward=2048` too large for `d_model=128` | Set to 512 |
| 9 | `video_paths` pointed to directory | `glob` for video files + `FileNotFoundError` |
| 10 | No validation or metrics | Val loop + MAE BPM + best-model checkpoint |

## v3 UBFC-rPPG Additions (inherited)
| # | Addition | Detail |
|---|----------|--------|
| 11 | **GPU/CPU auto-detection** | Explicit device check; CPU fallback with reduced defaults |
| 12 | **`load_ubfc_gt()`** | Loads BVP from `ground_truth.txt`; resamples if frame-count differs |
| 13 | **`process_video()` refactored** | Returns frames + `n_valid_frames` only (no POS) |
| 14 | **`RPPGDataset` updated** | Accepts `(video_path, gt_path)` pairs; real GT used for training |
| 15 | **UBFC path discovery** | Auto-discovers all `vid.avi` + `ground_truth.txt` pairs |
| 16 | **CPU-adaptive config** | Smaller batch, fewer default epochs, grad-ckpt disabled |

## Expected Healthy Training Behaviour After Fixes
- **MAE BPM epoch 1**: 20–40 BPM (random model — should NOT be 0.00 or 6.67)
- **MAE BPM trend**: should decrease monotonically toward <10 BPM within 20–30 epochs on GPU
- **Val loss**: should track train loss with small gap; divergence = overfitting (add weight decay / dropout)
- **Per-subject BPM table**: printed every epoch — use it to verify predictions are diverse (not stuck at one BPM)
- **Healthy final MAE**: <5 BPM on UBFC-rPPG dataset2 is achievable with this architecture
